# Self-Model Benchmark Evaluation

Validate self-model vs world-model on real time-series benchmarks.

**Goal**: Close the gap in proprietary_insights.md — self-model works on AR1 but is "unclear on complex" benchmarks (Weather, Traffic)

In [1]:
# Cell 1: Install dependencies
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "torch", "numpy", "pandas"])
print("✅ Dependencies installed")

✅ Dependencies installed


In [2]:
# Cell 2: Imports
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import io, requests, time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Quadro RTX 5000


In [3]:
# Cell 3: Benchmark configurations
BENCHMARKS = {
    "AR1": {"seq_len": 96, "pred_len": 24, "synthetic": True},  # Baseline
    "ETTh1": {"seq_len": 96, "pred_len": 24, "url": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv"},
    "ETTm1": {"seq_len": 96, "pred_len": 24, "url": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm1.csv"},
    "Weather": {"seq_len": 96, "pred_len": 24, "url": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/weather/weather.csv"},
    "Traffic": {"seq_len": 96, "pred_len": 24, "url": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/traffic/traffic.csv"},
}

In [4]:
# Cell 4: Load benchmark data
def load_benchmark(name, cfg):
    """Load benchmark data, normalize, split into train/val/test."""
    print(f"Loading {name}...", end=" ", flush=True)
    
    if cfg.get("synthetic"):
        # AR1-style synthetic data with autoregressive patterns
        n_samples = 5000
        t = np.linspace(0, 100, n_samples)
        # Complex AR(1) with multiple frequencies
        data = np.column_stack([
            0.8 * np.sin(t) + 0.2 * np.random.randn(n_samples),
            0.6 * np.cos(2*t) + 0.15 * np.random.randn(n_samples),
            0.4 * np.sin(t/3) * np.cos(t/5) + 0.1 * np.random.randn(n_samples),
            0.5 * (np.sin(t) > 0).astype(float) + 0.1 * np.random.randn(n_samples),
        ])
    else:
        try:
            df = pd.read_csv(cfg["url"])
            data = df.iloc[:, 1:].values  # Remove timestamp
        except Exception as e:
            print(f"failed: {e}")
            return None
    
    # Normalize
    mean, std = data.mean(axis=0), data.std(axis=0) + 1e-8
    data = (data - mean) / std
    
    # Split 70/10/20
    n = len(data)
    train, val, test = data[:int(0.7*n)], data[int(0.7*n):int(0.8*n)], data[int(0.8*n):]
    print(f"train={len(train)}, val={len(val)}, test={len(test)}")
    return train, val, test

benchmark_data = {}
for name, cfg in BENCHMARKS.items():
    data = load_benchmark(name, cfg)
    if data is not None:
        benchmark_data[name] = (*data, cfg["seq_len"], cfg["pred_len"])

print(f"\n✅ Loaded {len(benchmark_data)} benchmarks: {list(benchmark_data.keys())}")

Loading AR1... train=3500, val=500, test=1000
Loading ETTh1... train=3500, val=500, test=1000
Loading ETTh1... train=12194, val=1742, test=3484
Loading ETTm1... train=48776, val=6968, test=13936
Loading Weather... failed: HTTP Error 404: Not Found
Loading Traffic... failed: HTTP Error 404: Not Found

✅ Loaded 3 benchmarks: ['AR1', 'ETTh1', 'ETTm1']


In [5]:
# Cell 5: Self-Model and World-Model architectures

class SelfModelForecaster(nn.Module):
    """
    Self-model: Predict self-state first, then use it for forecasting.
    
    Core idea from COG-research:
    - Maintains internal state that models its own dynamics
    - Predicts own state before predicting environment
    - Spectral radius ≈ 1 enables near-critical dynamics
    """
    def __init__(self, d_input, d_model=64, d_state=32):
        super().__init__()
        self.d_input = d_input
        
        # Input encoder
        self.encoder = nn.Linear(d_input, d_model)
        
        # Self-model: GRU (batched) - much faster than GRUCell
        self.self_dynamics = nn.GRU(d_model, d_state, batch_first=True)
        
        # Self-predictor: predicts next self-state
        self.self_predictor = nn.Linear(d_state, d_state)
        
        # Forecasting head uses BOTH self-state AND input
        self.forecast_head = nn.Sequential(
            nn.Linear(d_model + d_state, d_model),
            nn.Tanh(),
            nn.Linear(d_model, 1)
        )
        
        self.d_state = d_state
        
    def forward(self, x, return_state=False):
        """
        x: (B, T, d_input)
        """
        B, T, D = x.shape
        
        # Encode all inputs at once (batch operation)
        enc = self.encoder(x)  # (B, T, d_model)
        
        # Self-model: process entire sequence with GRU (batch operation)
        h0 = torch.zeros(1, B, self.d_state, device=x.device)
        h, _ = self.self_dynamics(enc, h0)  # (B, T, d_state)
        
        # Self-model prediction error: predict h[t] from h[t-1]
        h_detached = h.detach()
        h_pred = self.self_predictor(h_detached[:, :-1, :])  # (B, T-1, d_state)
        h_current = h_detached[:, 1:, :]
        self_error = (h_pred - h_current).pow(2).mean()
        
        # Forecast: use self-model state + current input
        pred = self.forecast_head(torch.cat([enc, h], dim=-1))
        
        if return_state:
            return pred, self_error.item()
        return pred


class WorldModelForecaster(nn.Module):
    """
    World-model: Predict environment directly, without self-model.
    
    Baseline: Standard encoder-decoder that directly models environment.
    """
    def __init__(self, d_input, d_model=64, d_state=32):
        super().__init__()
        self.d_input = d_input
        
        # Input encoder
        self.encoder = nn.Linear(d_input, d_model)
        
        # World-model: GRU (batched) - much faster than GRUCell
        self.world_dynamics = nn.GRU(d_model, d_state, batch_first=True)
        
        # No self-model - just predict environment
        self.forecast_head = nn.Sequential(
            nn.Linear(d_model + d_state, d_model),
            nn.Tanh(),
            nn.Linear(d_model, 1)
        )
        
        self.d_state = d_state
        
    def forward(self, x):
        B, T, D = x.shape
        
        # Encode all inputs at once (batch operation)
        enc = self.encoder(x)  # (B, T, d_model)
        
        # World-model: process entire sequence with GRU (batch operation)
        h0 = torch.zeros(1, B, self.d_state, device=x.device)
        h, _ = self.world_dynamics(enc, h0)  # (B, T, d_state)
        
        # Forecast using world state + input
        pred = self.forecast_head(torch.cat([enc, h], dim=-1))
        
        return pred

In [6]:
# Cell 6: Training and evaluation functions

def train_forecaster(model, train_data, val_data, d_input, epochs=50, lr=1e-3):
    """Train a forecaster on time-series data."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    seq_len = 96
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        n_batches = 0
        
        # Shuffle training data
        indices = np.arange(0, len(train_data) - seq_len - 1, seq_len)
        np.random.shuffle(indices)
        
        for i in indices:
            x = torch.tensor(train_data[i:i+seq_len], dtype=torch.float32).unsqueeze(0).to(device)
            y = torch.tensor(train_data[i+1:i+seq_len+1, 0], dtype=torch.float32).unsqueeze(0).to(device)
            
            pred = model(x)
            loss = criterion(pred[:, :, 0], y)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
            n_batches += 1
    
    return model


def evaluate_forecaster(model, test_data, seq_len=96, pred_len=24):
    """Evaluate MSE on test set (multi-step forecasting)."""
    model.eval()
    total_mse = 0.0
    n_samples = 0
    
    with torch.no_grad():
        # Slide window across test data
        step = pred_len  # Non-overlapping windows
        for i in range(0, len(test_data) - seq_len - pred_len, step):
            x = torch.tensor(test_data[i:i+seq_len], dtype=torch.float32).unsqueeze(0).to(device)
            y_true = test_data[i+seq_len:i+seq_len+pred_len, 0]  # First feature
            
            # Train on context, predict future
            pred = model(x)
            pred = pred[0, -pred_len:, 0].cpu().numpy()
            
            mse = ((pred - y_true) ** 2).mean()
            total_mse += mse
            n_samples += 1
    
    return total_mse / max(n_samples, 1)


print("✅ Training and evaluation functions defined!")

✅ Training and evaluation functions defined!


In [7]:
# Cell 7: Run benchmark comparison

d_model = 64
d_state = 32
results = {}

print("="*70)
print("SELF-MODEL vs WORLD-MODEL BENCHMARK EVALUATION")
print("="*70)

for name, (train, val, test, seq_len, pred_len) in benchmark_data.items():
    print(f"\n{'='*50}")
    print(f"Benchmark: {name}")
    print(f"{'='*50}")
    
    d_input = train.shape[1]
    
    # Combine train+val for training
    combined = np.concatenate([train, val], axis=0)
    
    # Train self-model
    print(f"Training Self-Model ({d_model}d, {d_state}state)...", end=" ", flush=True)
    t0 = time.time()
    sm = SelfModelForecaster(d_input, d_model, d_state)
    sm = train_forecaster(sm, combined, val, d_input, epochs=50)
    sm_time = time.time() - t0
    print(f"done ({sm_time:.1f}s)")
    
    # Evaluate self-model
    sm_mse = evaluate_forecaster(sm, test, seq_len, pred_len)
    
    # Train world-model
    print(f"Training World-Model ({d_model}d, {d_state}state)...", end=" ", flush=True)
    t0 = time.time()
    wm = WorldModelForecaster(d_input, d_model, d_state)
    wm = train_forecaster(wm, combined, val, d_input, epochs=50)
    wm_time = time.time() - t0
    print(f"done ({wm_time:.1f}s)")
    
    # Evaluate world-model
    wm_mse = evaluate_forecaster(wm, test, seq_len, pred_len)
    
    # Calculate improvement
    if wm_mse > 0:
        improvement = (wm_mse - sm_mse) / wm_mse * 100
    else:
        improvement = 0
    
    print(f"\n  📊 Results:")
    print(f"     Self-Model MSE: {sm_mse:.6f}")
    print(f"     World-Model MSE: {wm_mse:.6f}")
    print(f"     Improvement: {improvement:+.1f}%")
    
    if improvement > 0:
        print(f"     ✅ Self-model wins!")
    else:
        print(f"     ❌ World-model wins")
    
    results[name] = {"self_model": sm_mse, "world_model": wm_mse, "improvement": improvement}

SELF-MODEL vs WORLD-MODEL BENCHMARK EVALUATION

Benchmark: AR1
Training Self-Model (64d, 32state)... done (7.4s)
Training World-Model (64d, 32state)... done (6.0s)

  📊 Results:
     Self-Model MSE: 0.301342
     World-Model MSE: 0.310371
     Improvement: +2.9%
     ✅ Self-model wins!

Benchmark: ETTh1
Training Self-Model (64d, 32state)... done (23.2s)
Training World-Model (64d, 32state)... done (21.7s)

  📊 Results:
     Self-Model MSE: 0.771868
     World-Model MSE: 0.754337
     Improvement: -2.3%
     ❌ World-model wins

Benchmark: ETTm1
Training Self-Model (64d, 32state)... done (96.9s)
Training World-Model (64d, 32state)... done (91.1s)

  📊 Results:
     Self-Model MSE: 3.041251
     World-Model MSE: 3.005511
     Improvement: -1.2%
     ❌ World-model wins


In [8]:
# Cell 8: Summary and comparison to proprietary insights

print("\n" + "="*70)
print("BENCHMARK SUMMARY")
print("="*70)

print(f"\n{'Benchmark':<12} | {'Self-Model':>12} | {'World-Model':>12} | {'Delta':>10}")
print("-"*50)
for name, r in results.items():
    delta = "+" if r["improvement"] > 0 else ""
    print(f"{name:<12} | {r['self_model']:>12.6f} | {r['world_model']:>12.6f} | {delta}{r['improvement']:>9.1f}%")

print("-"*50)

# Compare to proprietary insights claims
print("\n📋 Comparison to proprietary_insights.md:")
print("   - AR1: Claims 44% better (self-model: 0.010 vs world-model: 0.018)")
if "AR1" in results:
    ar1_delta = results["AR1"]["improvement"]
    print(f"   - Our AR1 result: {ar1_delta:+.1f}% (matching claim? {'✅' if ar1_delta > 30 else '⚠️'})")

print("\n🔍 Key insight from proprietary_insights.md:")
print("   'Self-model for forecasting: ✅ AR1, ETTh1, ETTm1 | ? Weather, Traffic'")
print("   → This benchmark validates the '?' for Weather and Traffic")

# Check if we're validating or invalidating the claim
weather_ok = results.get("Weather", {}).get("improvement", -999) > 0
traffic_ok = results.get("Traffic", {}).get("improvement", -999) > 0
etth_ok = results.get("ETTh1", {}).get("improvement", -999) > 0

print(f"\n🎯 Validation Status:")
print(f"   - AR1: {'✅ Validated' if results.get('AR1', {}).get('improvement', 0) > 0 else '❌ Not validated'}")
print(f"   - ETTh1: {'✅ Validated' if etth_ok else '❌ Not validated'}")
print(f"   - Weather: {'✅ Validated' if weather_ok else '❌ Not validated'}")
print(f"   - Traffic: {'✅ Validated' if traffic_ok else '❌ Not validated'}")


BENCHMARK SUMMARY

Benchmark    |   Self-Model |  World-Model |      Delta
--------------------------------------------------
AR1          |     0.301342 |     0.310371 | +      2.9%
ETTh1        |     0.771868 |     0.754337 |      -2.3%
ETTm1        |     3.041251 |     3.005511 |      -1.2%
--------------------------------------------------

📋 Comparison to proprietary_insights.md:
   - AR1: Claims 44% better (self-model: 0.010 vs world-model: 0.018)
   - Our AR1 result: +2.9% (matching claim? ⚠️)

🔍 Key insight from proprietary_insights.md:
   'Self-model for forecasting: ✅ AR1, ETTh1, ETTm1 | ? Weather, Traffic'
   → This benchmark validates the '?' for Weather and Traffic

🎯 Validation Status:
   - AR1: ✅ Validated
   - ETTh1: ❌ Not validated
   - Weather: ❌ Not validated
   - Traffic: ❌ Not validated


## Next Steps

After running this notebook:

1. **If self-model wins on Weather/Traffic**: Close the gap in proprietary_insights.md — update the '?' to '✅'
2. **If world-model wins**: Investigate why — possible causes:
   - Self-model needs larger scale (current: 64d)
   - Spectral radius not properly enforced
   - Task complexity differs from AR1
3. **Add anchor bottleneck**: Next notebook cell for long-context efficiency testing